# Training Models with their Best Params

## Setup

### Imports

In [ ]:
from typing import Optional, Dict, Any, Callable, Type

import pickle
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.base import BaseEstimator
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

### Load Datasets

In [ ]:
with open("../datasets/house_prices.pkl", mode="rb") as f:
    X_house_prices_train, X_house_prices_test, y_house_prices_train, y_house_prices_test = pickle.load(f)

X_house_prices = np.concatenate((X_house_prices_train, X_house_prices_test), axis=0)
y_house_prices = np.concatenate((y_house_prices_train, y_house_prices_test), axis=0)

print(X_house_prices.shape)
print(y_house_prices.shape)

### Auxiliar Functions

In [ ]:
def build_regressor_model(
    estimator_class: Type[BaseEstimator],
    model_params: Optional[Dict[str, Any]] = {},
    scaler: Optional[BaseEstimator] = None,
    func: Optional[Callable[[np.ndarray], np.ndarray]] = None,
    inverse_func: Optional[Callable[[np.ndarray], np.ndarray]] = None,
) -> BaseEstimator:
    estimator = estimator_class(**model_params)
    
    if scaler is not None:
        regressor_pipeline = Pipeline([
            ('scaler', scaler),
            ('model', estimator)
        ])
    else:
        regressor_pipeline = estimator
        
    if func is not None and inverse_func is not None:
        model = TransformedTargetRegressor(
            regressor=regressor_pipeline,
            func=func,
            inverse_func=inverse_func
        )
    else:
        model = regressor_pipeline
        
    return model

def get_estimator_model_name(model:BaseEstimator) -> str:
    if hasattr(model, "regressor"):
        model = model.regressor
        
    if isinstance(model, Pipeline):
        model = model.steps[-1][1]
        
    return model.__class__.__name__


In [ ]:
def apply_cross_validations_in_models(
    models: list[BaseEstimator], 
    X: np.typing.ArrayLike, 
    y: np.typing.ArrayLike,
    n_splits: int = 10,
    n_tests: int = 30
) -> pd.DataFrame:
    results: dict[str, list[float]] = {
        get_estimator_model_name(model): [] 
        for model in models
    }

    for i in tqdm(range(n_tests), desc="Tests", position=0):
        k_fold = KFold(n_splits=n_splits, shuffle=True, random_state=i)

        for model in models:
            scores = cross_val_score(
                estimator=model, 
                X=X, 
                y=y, 
                cv=k_fold, 
                scoring="neg_mean_absolute_error"
            )
            
            results[get_estimator_model_name(model)].append(-scores.mean())

    return pd.DataFrame(results)

## Training

### Models Array

In [ ]:
model_list:list[BaseEstimator] = [
    build_regressor_model(
        estimator_class=LinearRegression
    ),
    build_regressor_model(
        estimator_class=DecisionTreeRegressor, 
        model_params={'criterion': 'absolute_error', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}
    ),
    build_regressor_model(
        estimator_class=RandomForestRegressor, 
        model_params={'max_depth': 20, 'max_features': 0.5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
    ),
    build_regressor_model(
        estimator_class=SVR, 
        model_params={'C': 10, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf'}, 
        scaler=StandardScaler(),
        func=np.log1p,
        inverse_func=np.expm1
    ),
    build_regressor_model(
        estimator_class=MLPRegressor, 
        model_params={'activation': 'relu', 'alpha': 0.01, 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.001, 'max_iter': 500, 'solver': 'adam'},
        scaler=StandardScaler(),
        func=np.log1p,
        inverse_func=np.expm1
    )
]

### Multiple Cross Validations

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    df = apply_cross_validations_in_models(
        models=model_list, 
        X=X_house_prices, 
        y=y_house_prices,
        n_splits=5,
        n_tests=25
    )

    display(df)
    df.to_csv('results.csv', index=False)